# EasyRAG Evidence selection and top-k for answering

## Chapter position

Retrieval may return a decent ranked list, but answer generation still has to live inside a context budget. This notebook isolates that selection step.

## Learning question

How does EasyRAG decide which retrieved citations survive into the answer prompt?

## Success criteria

- You can explain what `normalize_citation_snippet()` does before budgeting starts.
- You can describe the item budget and character budget as two separate stop conditions.
- You can explain why the first citation may still be kept even when it already exceeds the character budget.

## Source code anchors

- `easyrag.rag.generation.selection.normalize_citation_snippet`
- `easyrag.rag.generation.selection.select_answer_citations`
- `easyrag.rag.types.AnswerParam`
- `easyrag.rag.types.QueryResult`

## Method principles

- `easyrag.rag.generation.selection.normalize_citation_snippet`: This helper cleans citation snippets into a compact, readable form before budgets and prompts see them. It removes noise without trying to rewrite the evidence semantically.
- `easyrag.rag.generation.selection.select_answer_citations`: This is the answer-side evidence budgeter. It walks citations in order and keeps a compact subset that fits the item and character limits.
- `easyrag.rag.types.AnswerParam`: This dataclass is the answer-side policy bundle. It controls citation budget, context budget, answer style, and abstention rules without changing retrieval behavior.
- `easyrag.rag.types.QueryResult`: This is the retrieval handoff object. It keeps chunks, citations, relations, and metadata together so later generation or evaluation code can inspect evidence instead of starting from raw storage records.

Design reason: these anchors break answering into evidence selection, packing, prompting, and synthesis so the notebook can show that answer quality is a data-shaping problem as much as a model-wording problem.


## How the code fits together

Selection sits between retrieval and prompting. `normalize_citation_snippet()` cleans citation text into a readable, compact form before the budget logic looks at it. `select_answer_citations()` then walks the citations in order and keeps adding them until the next item would break the answer budget. The order matters because the function does not reshuffle or rerank citations. It also stops on the first over-budget item after something has already been kept, which means later citations are never visited.

Design reason: the notebook walks the same evidence through smaller answer-stage boundaries before it asks for a final sentence. That ordering is what lets you see whether a failure came from evidence budget, context packing, prompting policy, or synthesis.


## What this cell is proving

We can use the real selection code and add one notebook-only trace helper around it. The helper does not replace the library function. It only spells out why the function made each decision.

In [ ]:
import importlib
import sys
import tempfile
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "notebooks" / "_utils.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError("Could not locate the EasyRAG repository root.")

_notebook_utils = importlib.import_module("notebooks._utils")
_rag = importlib.import_module("easyrag.rag")
_selection = importlib.import_module("easyrag.rag.generation.selection")
_async_utils = importlib.import_module("easyrag.support.async_utils")

ensure_repo_root_on_path = _notebook_utils.ensure_repo_root_on_path
_print_json = _notebook_utils.print_json
can_use_openai_compatible_models = _notebook_utils.provider_ready
skip_message = _notebook_utils.skip_message
_stub_embedding = _notebook_utils.stub_embedding
_stub_query_model = _notebook_utils.stub_query_model
AnswerParam = _rag.AnswerParam
EasyRAG = _rag.EasyRAG
QueryParam = _rag.QueryParam
normalize_citation_snippet = _selection.normalize_citation_snippet
select_answer_citations = _selection.select_answer_citations
run_sync = _async_utils.run_sync

REPO_ROOT = ensure_repo_root_on_path()


def trace_selection(
    citations: list[dict], *, max_items: int, max_chars: int
) -> tuple[list[dict], list[dict], int]:
    selected: list[dict] = []
    decisions: list[dict] = []
    remaining_budget = max_chars

    for citation in citations:
        normalized_snippet = normalize_citation_snippet(citation["snippet"])
        snippet_length = len(normalized_snippet)
        would_exceed_budget = bool(selected) and snippet_length > remaining_budget
        max_items_reached = len(selected) >= max_items

        decision = {
            "location": citation["location"],
            "snippet_chars": snippet_length,
            "remaining_before": remaining_budget,
        }

        if max_items_reached:
            decision["decision"] = "drop:max_items"
            decisions.append(decision)
            continue

        if would_exceed_budget:
            decision["decision"] = "drop:max_chars"
            decisions.append(decision)
            continue

        selected.append(citation)
        remaining_budget -= snippet_length
        decision["decision"] = "keep"
        decision["remaining_after"] = remaining_budget
        decisions.append(decision)

    return selected, decisions, remaining_budget

This cell should stay quiet. The only new idea here is the trace helper, which mirrors the library loop closely enough to explain the budget decisions.

## What this cell is proving

Retrieval can surface more evidence than the answer stage should keep. We want a raw list that is good enough to read, but still crowded enough that the answer budget has to make a real cut.

Design reason: this cell crosses the boundary from prepared inputs into ranked evidence. The notebook uses the structured retrieval APIs on purpose so citations, mode choice, and query metadata stay inspectable instead of disappearing behind a single search string.


In [ ]:
selection_tmp = tempfile.TemporaryDirectory()
selection_rag = EasyRAG(
    working_dir=selection_tmp.name,
    workspace="selection-demo",
    embedding_func=_stub_embedding,
    query_model_func=_stub_query_model,
)
run_sync(selection_rag.initialize_storages())
run_sync(
    selection_rag.ainsert(
        [
            "# Retrieval\nEasyRAG uses grounded retrieval and query rewrite to keep evidence traceable.\n",
            "# Storage\nRetrieved evidence is packed before answer synthesis and storage review.\n",
            "# Policy\nCitation-aware answers expose the evidence budget directly to readers.\n",
        ],
        ids=["doc::retrieval", "doc::storage", "doc::policy"],
        file_paths=["docs/retrieval.md", "docs/storage.md", "docs/policy.md"],
    )
)
selection_result = run_sync(
    selection_rag.aquery(
        "How does EasyRAG keep answers grounded?",
        QueryParam(mode="hybrid", chunk_top_k=5),
    )
)

_print_json(
    [
        {
            "location": citation["location"],
            "normalized_chars": len(
                normalize_citation_snippet(citation["snippet"])[:220]
            ),
            "snippet": normalize_citation_snippet(citation["snippet"]),
        }
        for citation in selection_result.citations
    ]
)

## Why this output looks like this

The citations are still in retrieval order. The selector will preserve that order. It does not look for a shorter citation later in the list once an earlier one would break the budget.

Design reason: the output looks layered because the same evidence is being carried through several answer-stage representations. Keeping the citation subset, packed context, prompt text, and final answer close together is what exposes whether a failure came from selection, packing, instruction design, or synthesis.


## What this cell is proving

The budget logic is plain enough to inspect directly. We will run the real selector and the trace helper side by side so the kept and dropped evidence has a concrete reason attached to it.

Design reason: generation is intentionally split into evidence selection, packing, prompting, and synthesis because each boundary can fail for a different reason. Keeping this step explicit is what makes answer quality debuggable instead of treating the model response as one indivisible artifact.


In [ ]:
tight_param = AnswerParam(max_citations=2, max_context_chars=120)
loose_param = AnswerParam(max_citations=4, max_context_chars=320)

tight_selection = select_answer_citations(
    selection_result.citations,
    max_items=tight_param.max_citations,
    max_chars=tight_param.max_context_chars,
)
tight_trace, tight_decisions, tight_budget = trace_selection(
    selection_result.citations,
    max_items=tight_param.max_citations,
    max_chars=tight_param.max_context_chars,
)

loose_selection = select_answer_citations(
    selection_result.citations,
    max_items=loose_param.max_citations,
    max_chars=loose_param.max_context_chars,
)
loose_trace, loose_decisions, loose_budget = trace_selection(
    selection_result.citations,
    max_items=loose_param.max_citations,
    max_chars=loose_param.max_context_chars,
)

_print_json(
    {
        "tight_budget": {
            "kept_locations": [citation["location"] for citation in tight_selection],
            "final_budget": tight_budget,
            "decisions": tight_decisions,
        },
        "loose_budget": {
            "kept_locations": [citation["location"] for citation in loose_selection],
            "final_budget": loose_budget,
            "decisions": loose_decisions,
        },
    }
)

## Why this output looks like this

The trace is usually more interesting than the final selected list. You can see the exact point where the loop stops, plus the citations that never even got a chance because an earlier one already exhausted the budget.

Design reason: the output looks layered because the same evidence is being carried through several answer-stage representations. Keeping the citation subset, packed context, prompt text, and final answer close together is what exposes whether a failure came from selection, packing, instruction design, or synthesis.


## What this cell is proving

The first citation is special. If the selected set is still empty, the function keeps one readable citation even when that single citation already exceeds `max_chars`. That behavior is deliberate. An empty context block is usually less useful than one over-budget citation you can still inspect.

Design reason: generation is intentionally split into evidence selection, packing, prompting, and synthesis because each boundary can fail for a different reason. Keeping this step explicit is what makes answer quality debuggable instead of treating the model response as one indivisible artifact.


In [ ]:
synthetic_citations = [
    {
        "title": "Long note",
        "location": "docs/long.md",
        "snippet": "Long evidence " * 30,
    },
    {
        "title": "Short note",
        "location": "docs/short.md",
        "snippet": "Short evidence that would fit on its own.",
    },
]

synthetic_selection = select_answer_citations(
    synthetic_citations, max_items=2, max_chars=40
)
_, synthetic_decisions, synthetic_budget = trace_selection(
    synthetic_citations,
    max_items=2,
    max_chars=40,
)

_print_json(
    {
        "selected": synthetic_selection,
        "final_budget": synthetic_budget,
        "decisions": synthetic_decisions,
    }
)

## Why this output looks like this

Notice the missing `selected and ...` guard in the first iteration. The selector only starts enforcing the hard stop after it already has something to return. That detail is easy to miss if you only look at the function signature.

Design reason: the output looks layered because the same evidence is being carried through several answer-stage representations. Keeping the citation subset, packed context, prompt text, and final answer close together is what exposes whether a failure came from selection, packing, instruction design, or synthesis.


## What this cell is proving

The provider-backed path still uses the same local selection logic after retrieval. That means the selection contract should stay stable even when the retriever itself changes.

In [ ]:
if not can_use_openai_compatible_models():
    print(skip_message("provider"))
else:
    provider_tmp = tempfile.TemporaryDirectory()
    provider_rag = EasyRAG(
        working_dir=provider_tmp.name, workspace="provider-selection-demo"
    )
    try:
        run_sync(provider_rag.initialize_storages())
        run_sync(
            provider_rag.ainsert(
                [
                    "# Retrieval\nGrounded retrieval feeds answer selection.\n",
                    "# Policy\nCitation-aware outputs preserve traceability.\n",
                ],
                ids=["doc::retrieval", "doc::policy"],
                file_paths=["docs/retrieval.md", "docs/policy.md"],
            )
        )
        provider_result = run_sync(
            provider_rag.aquery(
                "How does EasyRAG preserve traceability?",
                QueryParam(mode="hybrid", chunk_top_k=3),
            )
        )
        provider_selection = select_answer_citations(
            provider_result.citations,
            max_items=2,
            max_chars=160,
        )
        _print_json(provider_selection)
    except Exception as exc:
        print(f"Provider-backed path was skipped due to runtime error: {exc}")
    finally:
        try:
            run_sync(provider_rag.finalize_storages())
        finally:
            provider_tmp.cleanup()

## Common mistakes / debugging cues

- Do not treat retrieval `chunk_top_k` and answer `max_citations` as the same knob. They control different stages.
- Look at normalized snippet length, not raw snippet length, when a budget cut feels surprising.
- Once the selector stops, later citations are never reconsidered. If a later citation would have fit, the only fix is to change retrieval order or selection policy.

In [ ]:
run_sync(selection_rag.finalize_storages())
selection_tmp.cleanup()
print("Cleaned up the evidence-selection workspace.")

## Takeaway

Answer-side top-k is really a budget policy. The selector keeps citations in retrieval order, normalizes them, and stops as soon as the next addition would push the answer prompt past its budget.

Continue with [05_03_context_assembly_and_packing.ipynb](05_03_context_assembly_and_packing.ipynb) to see how this selected list becomes prompt-ready text.